# 03_model.ipynb — Part 3: Machine Learning
Uses `appointments_clean.csv` from Part 1 (sorted by appointment_id). Rows are never shuffled or re-sorted after loading.

In [28]:
!gdown 1rSzDn2NpfdyaWAyYv1iAPqi5hl6ygyYg

Downloading...
From: https://drive.google.com/uc?id=1rSzDn2NpfdyaWAyYv1iAPqi5hl6ygyYg
To: /content/appointments_clean.csv
100% 88.1k/88.1k [00:00<00:00, 63.9MB/s]


In [2]:
# 03_model.ipynb - setup (upload appointments_clean.csv first)
import numpy as np
import pandas as pd
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     cross_val_score, cross_val_predict)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             roc_auc_score)

## Task 3.1 — Features, split and baseline

In [3]:
# Task 3.1 - features, split, baseline
df = pd.read_csv("appointments_clean.csv")    # do NOT shuffle or re-sort

X = df[["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
        "reminder_sent", "booking_channel", "visit_type"]].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]                              # target: 1 = did not come

print("Number of features:", X.shape[1])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("No-shows in the test set:", int(y_test.sum()))

# baseline: always predict "Attended" (0)
baseline = (y_test == 0).mean()
print(f"Baseline accuracy: {baseline:.4f}",
      f"({int((y_test == 0).sum())} of {len(y_test)})")

Number of features: 11
Training rows: 792 | Test rows: 198
No-shows in the test set: 49
Baseline accuracy: 0.7525 (149 of 198)


## Task 3.2 — Logistic regression

In [4]:
# Task 3.2 - scale: fit on TRAINING data only, then apply to test
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
lr_pred = lr.predict(X_test_s)                 # 0 or 1 (cut-off 0.5)
lr_prob = lr.predict_proba(X_test_s)[:, 1]     # probability of no-show

print("Accuracy :", round(accuracy_score(y_test, lr_pred), 3))
print("Confusion matrix:\n", confusion_matrix(y_test, lr_pred))
print("Precision:", round(precision_score(y_test, lr_pred), 3))
print("Recall   :", round(recall_score(y_test, lr_pred), 3))
print("F1       :", round(f1_score(y_test, lr_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test, lr_prob), 3))

Accuracy : 0.788
Confusion matrix:
 [[139  10]
 [ 32  17]]
Precision: 0.63
Recall   : 0.347
F1       : 0.447
ROC-AUC  : 0.789


In [5]:
# coefficients next to the feature names (same order as X)
coefs = pd.DataFrame({"feature": X.columns, "coefficient": lr.coef_[0]})
print(coefs.round(3).to_string(index=False))

# reminder_sent: two ways to read its coefficient
i = list(X.columns).index("reminder_sent")
coef = lr.coef_[0][i]
sd = scaler.scale_[i]                  # standard deviation of reminder_sent
print(f"reminder_sent coefficient: {coef:.3f}")
print(f"SD of reminder_sent: {sd:.2f}")
print(f"np.exp(coef)      = {np.exp(coef):.2f}   per 1 SD")
print(f"np.exp(coef / sd) = {np.exp(coef / sd):.2f}   No -> Yes")
print(f"=> about {100 * (1 - np.exp(coef / sd)):.0f}% lower odds")

                feature  coefficient
                    age       -0.254
            distance_km        0.253
                fee_inr        0.453
              lead_days        0.614
         prior_no_shows        0.650
          reminder_sent       -0.581
    booking_channel_App       -0.047
  booking_channel_Phone        0.216
booking_channel_Walk-In       -0.186
visit_type_Consultation        0.016
   visit_type_Follow-up       -0.016
reminder_sent coefficient: -0.581
SD of reminder_sent: 0.49
np.exp(coef)      = 0.56   per 1 SD
np.exp(coef / sd) = 0.31   No -> Yes
=> about 69% lower odds


**Answer 3.2.** `np.exp(coef)` = 0.56 is the odds multiplier for one standard deviation of the scaled `reminder_sent` (SD = 0.49); `np.exp(coef / SD)` = 0.31 is the multiplier for a real change from No (0) to Yes (1). They differ because the model was fitted on scaled features, so the raw coefficient describes a step of one SD (0.49), not the full 0 → 1 step.

In this model, holding the other features constant, appointments with a reminder have about **69% lower odds** of a no-show. The data are observational, so this is an association: it does not prove that reminders cause fewer no-shows.

## Task 3.3 — Decision tree and overfitting

In [6]:
# Task 3.3 - decision tree with no depth limit
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)
print("Full tree  depth:", tree_full.get_depth(),
      "| train:", round(tree_full.score(X_train, y_train), 3),
      "| test:", round(tree_full.score(X_test, y_test), 3))

# the same tree limited to depth 3
tree3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree3.fit(X_train, y_train)
print("Depth-3 tree",
      "| train:", round(tree3.score(X_train, y_train), 3),
      "| test:", round(tree3.score(X_test, y_test), 3))

Full tree  depth: 15 | train: 1.0 | test: 0.672
Depth-3 tree | train: 0.822 | test: 0.778


**Answer 3.3.** The full tree scores 1.00 on the training set but only 0.672 on the test set (below the 0.7525 baseline), so it memorised the training data and overfits, while the depth-3 tree (0.822 train / 0.778 test) generalises better.

## Task 3.4 — Random forest and comparison table

In [7]:
# Task 3.4 - random forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]
print("Forest train:", round(rf.score(X_train, y_train), 3),
      "| test:", round(accuracy_score(y_test, rf_pred), 3),
      "| recall:", round(recall_score(y_test, rf_pred), 3),
      "| AUC:", round(roc_auc_score(y_test, rf_prob), 3))

importances = pd.Series(rf.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False).head(3).round(3))

# comparison table - for REPORTING only (the choice is made in 3.5)
tree3_pred = tree3.predict(X_test)
tree3_prob = tree3.predict_proba(X_test)[:, 1]
comparison = pd.DataFrame({
    "model": ["Logistic regression", "Tree depth 3", "Random forest"],
    "accuracy": [accuracy_score(y_test, p)
                 for p in (lr_pred, tree3_pred, rf_pred)],
    "recall": [recall_score(y_test, p)
               for p in (lr_pred, tree3_pred, rf_pred)],
    "roc_auc": [roc_auc_score(y_test, p)
                for p in (lr_prob, tree3_prob, rf_prob)],
}).round(3)
print(comparison.to_string(index=False))

Forest train: 1.0 | test: 0.747 | recall: 0.306 | AUC: 0.706
lead_days      0.220
age            0.197
distance_km    0.195
dtype: float64
              model  accuracy  recall  roc_auc
Logistic regression     0.788   0.347    0.789
       Tree depth 3     0.778   0.224    0.700
      Random forest     0.747   0.306    0.706


**Answer 3.4.** Logistic regression looks best: it has the highest test ROC-AUC (0.789), the highest recall (0.347) and the highest accuracy (0.788). This table is for reporting only — the model is chosen in Task 3.5 using training data only.

## Task 3.5 — Tune, choose the model and the threshold, then test once

In [8]:
# Task 3.5 (a) - tune the forest with 5-fold cross-validation
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
                    cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)                    # training set ONLY
print("best_params_:", grid.best_params_)
print("best_score_ :", round(grid.best_score_, 3))   # a CV ROC-AUC

best_params_: {'max_depth': 3, 'n_estimators': 200}
best_score_ : 0.767


In [9]:
# (b) choose the model with training data only
lr_pipe = Pipeline([("scaler", StandardScaler()),
                    ("model", LogisticRegression(max_iter=1000))])
models = {
    "Logistic regression": lr_pipe,               # gets UNSCALED X_train
    "Tree depth 3": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Tuned forest": RandomForestClassifier(random_state=42, **grid.best_params_),
}
cv_scores = {}
for name, model in models.items():
    if name == "Tuned forest":
        cv_scores[name] = grid.best_score_        # already cross-validated
    else:
        cv_scores[name] = cross_val_score(model, X_train, y_train, cv=5,
                                          scoring="roc_auc").mean()
    print(f"{name:20s} CV ROC-AUC = {cv_scores[name]:.3f}")

winner = max(cv_scores, key=cv_scores.get)
final_model = models[winner]
print("Winner:", winner)

Logistic regression  CV ROC-AUC = 0.774
Tree depth 3         CV ROC-AUC = 0.678


Random forest        CV ROC-AUC = 0.747
Tuned forest         CV ROC-AUC = 0.767
Winner: Logistic regression


**Answer 3.5(b).** Logistic regression has the highest 5-fold cross-validated ROC-AUC on the training set (0.774, against 0.767 for the tuned forest, 0.747 for the random forest and 0.678 for the depth-3 tree), so it is the final model. The pipeline re-fits the scaler inside every fold, so no validation fold influences the scaling.

In [10]:
# (c) choose the threshold with OUT-OF-FOLD training probabilities
oof = cross_val_predict(final_model, X_train, y_train, cv=5,
                        method="predict_proba")[:, 1]

rows = []
for t in [0.5, 0.35]:
    flag = (oof >= t).astype(int)
    rows.append({"threshold": t,
                 "precision": precision_score(y_train, flag),
                 "recall": recall_score(y_train, flag),
                 "f1": f1_score(y_train, flag),
                 "flagged": int(flag.sum())})
thr_table = pd.DataFrame(rows).round(3)
print(thr_table.to_string(index=False))
THRESHOLD = thr_table.loc[thr_table["f1"].idxmax(), "threshold"]
print("Chosen threshold:", THRESHOLD)

 threshold  precision  recall    f1  flagged
      0.50      0.575   0.249 0.347       73
      0.35      0.513   0.467 0.489      154
Chosen threshold: 0.35


**Answer 3.5(c).** Threshold 0.35 is chosen: its F1 is higher (0.489 against 0.347) and its recall is almost double (0.467 against 0.249). Recall matters here because a missed warning costs an empty doctor slot, while an extra phone call costs about a minute.

In [11]:
# (d) fit on the training set, then test ONCE
final_model.fit(X_train, y_train)            # the pipeline fits its scaler
test_prob = final_model.predict_proba(X_test)[:, 1]
test_flag = (test_prob >= THRESHOLD).astype(int)
print("Precision:", round(precision_score(y_test, test_flag), 3))
print("Recall   :", round(recall_score(y_test, test_flag), 3))
print("Flagged  :", int(test_flag.sum()), "of", len(y_test))
print("ROC-AUC  :", round(roc_auc_score(y_test, test_prob), 3))

# (e) save one row per test appointment
scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": test_prob.round(4),
    "flag": test_flag,
})
scores.to_csv("test_risk_scores.csv", index=False)
print("Saved test_risk_scores.csv:", scores.shape,
      "| flagged:", int(scores["flag"].sum()))

Precision: 0.571
Recall   : 0.571
Flagged  : 49 of 198
ROC-AUC  : 0.789
Saved test_risk_scores.csv: (198, 4) | flagged: 49


**Answer 3.5(d)–(e).** One test-set evaluation of logistic regression at 0.35: precision 0.571, recall 0.571, 49 of 198 appointments flagged, ROC-AUC 0.789. `test_risk_scores.csv` has 198 rows and 49 flags, the same number as reported.

A real clinic would choose the threshold by how many follow-up calls the front desk can make in an evening, weighed against the cost of an empty slot.